# GLM-HMM: How to initialize the parameters

To obtain the GLM-HMM parameters, we use the EM algorithm. This is good for a variety of reasons, but it comes with the all-too-known downside of being overly sensitive to the initial parameters that you start your fitting this.

Because of that, in this notebook, we will go through 3 different initialization methods for the algorithm and compare them. 

By the end of this tutorial you should be familiar with:

1. How to fit a GLM-HMM 
2. How to initialize parameters using: random initialization, perturbation with Gaussian noise and K-segmentation

## Questions
- Should I start with simulated data before diving into the real dataset?
- Testing of different initialization methods and benchmarking should occur independent of the reproduction of ashwood figures because ashwood's individual animal fitting is "biased"? because of the global glm and then global glm hmm fit parameters used as initializations for each animals -> use two different notebooks?
- For the ashwood reproduction, I should fit both globals in my machine and then store those parameters. Then get the notebook from there.

## Ideas for class
- a method for sampling might be useful

In [ ]:
np.array([[1, 2, 3]])

array([[1, 2, 3]])

In [47]:
n_timepoints = 100
rates = np.zeros((n_timepoints,))
rates

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## Setup

In [17]:
import nemos
import jax
import numpy as np

from nemos.fetch import fetch_data
from nemos.glm import GLM
from nemos.glm_hmm.expectation_maximization import (
    backward_pass,
    compute_xi,
    forward_backward,
    forward_pass,
    hmm_negative_log_likelihood,
    run_m_step,
)
from nemos.observation_models import BernoulliObservations
from nemos.third_party.jaxopt.jaxopt import LBFGS
import jax.numpy as jnp



## Utils

In [ ]:
# Utils
def random_sticky(
    n_states: int,
    random_key: jax.random.PRNGKey = jax.random.PRNGKey(123),
    prob_stay=0.95,
):
    """
    Initialize transition probabilities with sticky dynamics.

    Creates a transition probability matrix that favors staying in the current state.

    Parameters
    ----------
    n_states :
        Number of HMM states. Must be greater than 1.
    random_key :
        Random key, unused for this particular initialization, but added for API consistency.
    prob_stay :
        Probability of staying in the current state. Default is 0.95.

    Returns
    -------
    :
        Transition probability matrix of shape (n_states, n_states).
    """
    prob_stay = prob_stay * (0.8 + 0.2 * jax.random.uniform(random_key, shape=(1,)))

    # assume n_state is > 1
    prob_leave = (1 - prob_stay) / (n_states - 1)

    return jnp.full((n_states, n_states), prob_leave) + jnp.diag(
        (prob_stay - prob_leave) * jnp.ones(n_states)
    )



def prepare_partial_hmm_nll_single_neuron(obs):
    # Define nll vmap function
    negative_log_likelihood = jax.vmap(
        lambda x, z: obs._negative_log_likelihood(
            x, z, aggregate_sample_scores=lambda w: w
        ),
        in_axes=(None, 1),
        out_axes=1,
    )

    # Solver
    def partial_hmm_negative_log_likelihood(
        weights, design_matrix, observations, posterior_prob
    ):
        return hmm_negative_log_likelihood(
            weights,
            X=design_matrix,
            y=observations,
            posteriors=posterior_prob,
            inverse_link_function=obs.default_inverse_link_function,
            negative_log_likelihood_func=negative_log_likelihood,
        )

    solver = LBFGS(partial_hmm_negative_log_likelihood, tol=10**-13)

    return partial_hmm_negative_log_likelihood, solver

## Playground - Simulation: 
Aim: create simulation of task similar to the one in Ashwood. This to get a final LL which can be compared between different initialization techniques


Create the necessary ingredients for the simulation: transition matrix, initial probabilities and

In [24]:
transition_matrix = random_sticky(3)

print(transition_matrix)

[[0.80288327 0.09855837 0.09855837]
 [0.09855837 0.80288327 0.09855837]
 [0.09855837 0.09855837 0.80288327]]


In [ ]:
jax.config.update("jax_enable_x64", True)                   
# Fetch the data
data_name =  "julia_regression_mstep_no_prior.npz"
data_path = fetch_data(data_name)
data = np.load(data_path)

# Design matrix and observed choices
X, y = data["X"], data["y"]

# Dirichlet priors
dirichlet_prior_initial_prob = data[
    "dirichlet_prior_initial_prob"
]  # storing alphas (NOT alphas + 1)
dirichlet_prior_transition_prob = data[
    "dirichlet_prior_transition_prob"
]  # storing alphas (NOT alphas + 1)

# M-step input
gammas = data["gammas"]
xis = data["xis"]
projection_weights = data["projection_weights"]
intercept, coef = projection_weights[:1], projection_weights[1:]
new_sess = data["new_sess"]

# M-step output
optimized_projection_weights = data["optimized_projection_weights"]
opt_intercept, opt_coef = (
    optimized_projection_weights[:1],
    optimized_projection_weights[1:],
)
new_initial_prob = data["new_initial_prob"]
new_transition_prob = data["new_transition_prob"]

In [15]:
# Initialize nemos observation model
obs = BernoulliObservations()

# Prepare nll function & solver
partial_hmm_negative_log_likelihood, solver = prepare_partial_hmm_nll_single_neuron(
    obs
)

(
    optimized_projection_weights_nemos,
    new_initial_prob_nemos,
    new_transition_prob_nemos,
    state,
) = run_m_step(
    X[:, 1:],  # drop intercept column
    y,
    gammas,
    xis,
    (coef, intercept),
    is_new_session=new_sess.astype(bool),
    solver_run=solver.run,
    dirichlet_prior_alphas_init_prob=dirichlet_prior_initial_prob,
    dirichlet_prior_alphas_transition=dirichlet_prior_transition_prob,
)

# NLL with nemos input
n_ll_nemos = partial_hmm_negative_log_likelihood(
    optimized_projection_weights_nemos,
    X[:, 1:],
    y,
    gammas,
)

# NLL with simulation input
n_ll_original = partial_hmm_negative_log_likelihood(
    (opt_coef, opt_intercept),
    X[:, 1:],
    y,
    gammas,
)

In [2]:
import jax.random

import nemos as nmo
import numpy as np
from nemos.glm_hmm import forward_backward
import jax.numpy as jnp

def random_sticky(
    n_states: int,
    random_key: jax.random.PRNGKey = jax.random.PRNGKey(123),
    prob_stay=0.95,
):
    """
    Initialize transition probabilities with sticky dynamics.

    Creates a transition probability matrix that favors staying in the current state.

    Parameters
    ----------
    n_states :
        Number of HMM states. Must be greater than 1.
    random_key :
        Random key, unused for this particular initialization, but added for API consistency.
    prob_stay :
        Probability of staying in the current state. Default is 0.95.

    Returns
    -------
    :
        Transition probability matrix of shape (n_states, n_states).
    """
    prob_stay = prob_stay * (0.8 + 0.2 * jax.random.uniform(random_key, shape=(1,)))

    # assume n_state is > 1
    prob_leave = (1 - prob_stay) / (n_states - 1)

    return jnp.full((n_states, n_states), prob_leave) + jnp.diag(
        (prob_stay - prob_leave) * jnp.ones(n_states)
    )

data_path = nmo.fetch.fetch_data(f"glm_hmm_simulation_n_neurons_5_seed_123.npz")
data = np.load(data_path)

# Design matrix and observed choices
X, counts = data["design_matrix"], data["counts"][:, 0]
latent_states = data["latent_states"]

# Initial parameters by:
# 1) fit a glm-hmm with random init condition
# 2) get a likely state per time point
# 3) fit one glm per state and store the coef and intercept
# 4) use the coef and intercept to initialize the GLMHMM
model = nmo.glm_hmm.GLMHMM(3,  initialize_transition_proba=random_sticky, regularizer="UnRegularized", regularizer_strength=100, tol=10**-8)
model.fit(X[:, 1:], counts)
(
    posteriors,
    joint_posterior,
    log_likelihood,
    log_likelihood_norm,
    alphas_noisy_params,
    betas_noisy_params,
) = forward_backward(
    X[:, 1:],  # drop intercept
    counts,
    model.initial_prob_,
    model.transition_prob_,
    model.glm_params_,
    likelihood_func=model._likelihood_func,
    inverse_link_function=model.inverse_link_function,
)
idx = np.argmax(posteriors, axis=1)
coef_init = np.zeros_like(model.coef_)
intercept = np.zeros_like(model.intercept_)


ValueError: File 'glm_hmm_simulation_n_neurons_5_seed_123.npz' is not in the registry.

In [ ]:

for i in np.unique(idx):
    glm = nmo.glm.GLM()
    glm.fit(X[idx==i, 1:], counts[idx == i])
    coef_init[:, i] = glm.coef_
    intercept[i] = glm.intercept_[i]
init_glm_params = (coef_init, intercept)


# fit with a bunch of different random seeds
arrays = []
for seed in range(0, 10):
    model = nmo.glm_hmm.GLMHMM(
        3,
        initialize_glm_params=init_glm_params,
        initialize_transition_proba=random_sticky,
        seed=jax.random.PRNGKey(seed+1000)
    )
    model.fit(X[:,1:], counts)
    arrays.append(model.coef_)

    (
        posteriors,
        joint_posterior,
        log_likelihood,
        log_likelihood_norm,
        alphas_noisy_params,
        betas_noisy_params,
    ) = forward_backward(
        X[:, 1:],  # drop intercept
        counts,
        model.initial_prob_,
        model.transition_prob_,
        model.glm_params_,
        likelihood_func=model._likelihood_func,
        inverse_link_function=model.inverse_link_function,
    )
    corr_matrix = np.corrcoef(latent_states.T, posteriors.T)[
        : latent_states.shape[1], latent_states.shape[1] :
    ]
    max_corr = np.max(corr_matrix, axis=1)

# Use first array as reference
ref = arrays[0].flatten()

print("Checking if each array is an affine transformation of the first array:")
print("(Looking for: array_i = a * array_0 + b)\n")

for i in range(1, len(arrays)):
    y = arrays[i].flatten()

    # Fit y = a * ref + b
    X = np.column_stack([ref, np.ones(len(ref))])
    coef = np.linalg.lstsq(X, y, rcond=None)[0]

    y_pred = X @ coef
    residuals = y - y_pred
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r_squared = 1 - (ss_res / ss_tot)

    print(f"Array {i}: R² = {r_squared:.6f}, RMSE = {np.sqrt(ss_res / len(y)):.6f}")

# Hide

## Data import
We will use the data from one mouse from Ashwood et al. (2020)

## Random initialization

## Utility function to iterate through different ones